<a href="https://colab.research.google.com/github/juanpajedrez/learn_rag_Huggingface/blob/main/learn_hf_object_detection_gradio_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gradio App

Now that we have finally uploaded our fine-tuned RT-DETR-v2 model to huggingface on trashify, the next step is to create our gradio app!

The following contains all the module preparation code in order to create the `Gradio` app with our trained model URL, found in:
https://huggingface.co/juanpajedrez/rt_detrv2_finetuned_trashify_box_detector_v1

In [1]:
!nvidia-smi

Wed Jul 29 21:21:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   43C    P8             16W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [11]:
# Install/import dependencies (this is mostly for Google Colab, as the other dependences are available by default in Colab)
try:
  import datasets
  import gradio as gr
  import torchmetrics
  import pycocotools
  import spaces
except ModuleNotFoundError:

  # If a module isn't found, install it
  !pip install -U datasets gradio spaces # -U stands for "upgrade" so we'll get the latest version by default
  !pip install -U torchmetrics[detection]

  import datasets
  import gradio as gr

  # Required for evalation
  import torchmetrics
  import pycocotools # make sure we have this for torchmetrics

import random

import numpy as np

import torch
import transformers

# Check versions (as long as you've got the following versions or higher, you should be good)
print(f"Using transformers version: {transformers.__version__}")
print(f"Using datasets version: {datasets.__version__}")
print(f"Using torch version: {torch.__version__}")
print(f"Using torchmetrics version: {torchmetrics.__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.6/111.6 kB 10.7 MB/s eta 0:00:00
Using transformers version: 5.13.1
Using datasets version: 5.0.1
Using torch version: 2.11.0+cu128
Using torchmetrics version: 1.9.0


In [3]:
from accelerate import Accelerator

accelerator = Accelerator(mixed_precision='fp16')

### Loading our model from Hugging Face

In [4]:
from transformers import AutoImageProcessor, AutoModelForObjectDetection

loaded_image_processor = AutoImageProcessor.from_pretrained("juanpajedrez/rt_detrv2_finetuned_trashify_box_detector_v1")
loaded_model = AutoModelForObjectDetection.from_pretrained("juanpajedrez/rt_detrv2_finetuned_trashify_box_detector_v1", device_map="auto")

preprocessor_config.json:   0%|          | 0.00/437 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.79k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  172MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

### Creating a demo of our model with Gradio

Gradio = one of the easiest ways to build ML apps.

And we'll host our app on Hugging Face Spaces.

Hugging Face Spaces = a place to share our app and model to the world.

1. Upload files manually.
2. Programmatically - https://huggingface.co/docs/huggingface_hub/package_reference/hf_api

We're going to go with option 2.
We are gonna create a file structure demo like the following:

```
demos/
└── trashify_object_detector/
    ├── app.py
    ├── README.md
    ├── requirements.txt
    └── trashify_examples/
        ├── trashify_example_1.jpeg
        ├── trashify_example_2.jpeg
        └── trashify_example_3.jpeg
```


In [5]:
loaded_model.config

RTDetrV2Config {
  "activation_dropout": 0.0,
  "activation_function": "silu",
  "anchor_image_size": null,
  "architectures": [
    "RTDetrV2ForObjectDetection"
  ],
  "attention_dropout": 0.0,
  "auxiliary_loss": true,
  "backbone": null,
  "backbone_config": {
    "depths": [
      3,
      4,
      6,
      3
    ],
    "downsample_in_bottleneck": false,
    "downsample_in_first_stage": false,
    "dtype": "float32",
    "embedding_size": 64,
    "hidden_act": "relu",
    "hidden_sizes": [
      256,
      512,
      1024,
      2048
    ],
    "layer_type": "bottleneck",
    "model_type": "rt_detr_resnet",
    "num_channels": 3,
    "out_features": [
      "stage2",
      "stage3",
      "stage4"
    ],
    "out_indices": [
      2,
      3,
      4
    ],
    "stage_names": [
      "stem",
      "stage1",
      "stage2",
      "stage3",
      "stage4"
    ]
  },
  "batch_norm_eps": 1e-05,
  "box_noise_scale": 1.0,
  "d_model": 256,
  "decoder_activation_function": "relu",
  "deco

In [6]:
from pathlib import Path

# Setup the demo path for our app files to live
demo_path = Path("../demos/trashify_object_detector/")

# Create the directory
demo_path.mkdir(parents=True, exist_ok=True)

### Making an `app.py` file

The `app.py` is the main file of our demo.

When we upload files to Hugging Face Spaces, Spaces will automatically look for the `app.py` file.

If it finds the file, it will run it as a Python Script.

Inside our `app.py` file, we'll have:
1. Import the required dependencies.
2. Setup preprocessing and model function. - `juanpajedrez/rt_detrv2_finetuned_trashify_box_detector_v1`.
3. Create a function `predict_on_image`:
  1. Take in an image, preprocess it.
  2. Predict on the image with our model.
  3. Post Process the predictions.
  4. Draw the predictions on the target image.
  5. Return the target image with the predictions drawn on.
4. We'll draw the model's predicted boxes on the image (if there are any).
5. Write some logic to display a string wether the target items (bin, thrash, hand) are present for +1 points.
6. We'll create a demo using Gradios `gr.Interface` class, this taks an input and produces an output:
https://gradio.app/docs/gradio/interface

This is in light of the Gradio Workflow:

Input (image) -> function (`predict_on_image`) -> output (`image`).

**Note:** As is 7/29/2026 -> Hugging Face Gradio demos can only be run on Zero GPUs for free users, therefore we are gonna do the code in such a way is on Zero GPU.

In [40]:
%%writefile ../demos/trashify_object_detector/app.py

# 1. Import necessary libraries
import gradio as gr
import torch
import spaces

from PIL import Image, ImageDraw, ImageFont
from transformers import AutoImageProcessor, AutoModelForObjectDetection

# Model path = juanpajedrez/rt_detrv2_finetuned_trashify_box_detector_v1

# 2. Setup preprocessing and model functions
model_save_path = "juanpajedrez/rt_detrv2_finetuned_trashify_box_detector_v1"

# Load image processor
loaded_image_processor = AutoImageProcessor.from_pretrained(model_save_path)

# Default size 640 x 640 for simplicity, also handles strange shaped images
loaded_image_processor.size = {"height": 640,
                               "width": 640}

# Load the model
loaded_model = AutoModelForObjectDetection.from_pretrained(model_save_path) # This is for Zero GPU or CPU

# Setup the target device (use GPU if it's available)
# Note: You can use Zero GPU free clusters from:
device = "cuda" if torch.cuda.is_available() else "cpu"
loaded_model = loaded_model.to(device)

# Get the id2label dictionary from the model
id2label = loaded_model.config.id2label

# Setup a color dictionary for pretty drawings
color_dict = {
    "bin": "green",
    "trash": "blue",
    "hand": "purple",
    "trash_arm": "yellow",
    "not_trash": "red",
    "not_bin": "red",
    "not_hand": "red"
}

# Use a GPU on a target function
@spaces.GPU
def predict_on_image(image, conf_threshold):
  loaded_model.eval()

  # Make a prediction on target image
  with torch.no_grad():
    inputs = loaded_image_processor(
        images=[image],
        return_tensors="pt"
    )
    model_outputs = loaded_model(**inputs.to(device))

    # Get original size of the image from PIL
    # PIL.image.size = width, height
    # post_process_object_detection is height by width
    target_sizes = torch.tensor([[image.size[1], image.size[0]]]) # -> [batch_size, height, width]

    # Post process our model outputs with image_processor.post_process_object_detection()
    results = loaded_image_processor.post_process_object_detection(
        outputs=model_outputs,
        threshold=conf_threshold,
        target_sizes=target_sizes
    )[0]

  # Return all data items/objects to the CPU if they aren't already there
  for key, value in results.items():
    try:
      results[key] = value.item().cpu() # Can't get scalars as .item() so add/try except block
    except:
      results[key] = value.cpu()

  ### 4. Draw the predictions on the target image ###
  draw = ImageDraw.Draw(image)

  # Get a font to write on our image
  font = ImageFont.load_default(size = 20)

  # Get a list of the detected class names
  detected_class_names_text_labels = []

  # Iterate through the predictions of the model and draw them on the arget image
  for box, label, score in zip(results["boxes"], results["labels"], results["scores"]):
    # Create the coordinates
    x, y, x2, y2 = tuple(box.tolist()) # XYXY

    # Get the text-based label names
    label_name = id2label[label.item()]
    target_color = color_dict[label_name]
    detected_class_names_text_labels.append(label_name)

    # Draw the bounding box
    draw.rectangle(xy = [x, y, x2, y2],
                   outline=target_color,
                   width = 3)

    # Create a text to display on the box
    text_string_to_show = f"{label_name} ({round(score.item(), 4)})"

    # Draw the text on the image
    draw.text(xy = (x, y),
              text = text_string_to_show,
              fill = "white",
              font = font)

  # Remove the draw each time to make sure it doesn't get caught in memory
  del draw

  ### 5. Create logic for outputting information message

  # Setup set of target items to discover
  target_items = {"trash", "bin", "hand"}
  detected_items = set(detected_class_names_text_labels)

  # If not items detected or bin, thrash, hand not in detected items, return notification
  if not detected_items & target_items:
    return_string = (
        f"No trash, bin, or hand detected at confidence threshold {conf_threshold}."
        "Try another image or lowering the confidence threshold."
    )
    print(return_string)
    return image, return_string

  # If there are items missing, output what;s missing for +1 point
  missing_items = target_items - detected_items
  if missing_items:
    return_string =  (
        f"Detected the following items: {sorted(detected_items & target_items)}."
        f"Missing the following: {missing_items}. "
        "In order to get +1 points, all target items must be detected"
    )
    print(return_string)
    return image, return_string

  # Final case, all items are detected
  return_string =  f"+1! Found the following items: {sorted(detected_items & target_items)}, thank you for cleaning up your local area!"
  print(return_string)
  return image, return_string

### 6. Setup the demo application to take in image/conf threshold, pass it through our function, show the output image/text

# Write description for our demo application
description = """
Help clean up your local area! Upload an image and get +1 if there is all of the following items detected: trash, bin, hand.

Model is a fine-tuned version of [RT-DETRv2](https://huggingface.co/docs/transformers/main/en/model_doc/rt_detr_v2#transformers.RTDetrV2Config) on the [Trashify dataset](https://huggingface.co/datasets/mrdbourke/trashify_manual_labelled_images).

See the full data loading and training code on [learnhuggingface.com](https://www.learnhuggingface.com/notebooks/hugging_face_object_detection_tutorial).

This is my video tutorial version :). Thank you mrdbourke for your teachings!
"""

# Creaate the gradio Interface
demo  = gr.Interface(
    fn = predict_on_image,
    inputs = [
        gr.Image(type = "pil", label = "Target Input Image"),
        gr.Slider(minimum = 0, maximum = 1, value = 0.1, label = "Confidence Threshold (Set higher for more confident boxes)")
    ],
    outputs = [
        gr.Image(type = "pil", label = "Target Image Output"),
        gr.Text(label = "Output Text")
    ],
    title = "Trashify Object Detector Video!",
    description = description,
    examples = [
        ["trashify_examples/trashify_example_0.jpeg", 0.1],
        ["trashify_examples/trashify_example_1.jpeg", 0.1],
        ["trashify_examples/trashify_example_2.jpeg", 0.1],
    ],
    cache_examples = True
    )

# Launch demo
#demo.launch(debug = True) # run with debug = True to see errors in google collab
demo.launch()

Overwriting ../demos/trashify_object_detector/app.py


### Making a `requirements.txt` file.

Dependencies for the file

In [44]:
%%writefile ../demos/trashify_object_detector/requirements.txt
timm
gradio
torch
transformers
spaces

Writing ../demos/trashify_object_detector/requirements.txt


### Making a `README.md` file.

In [25]:
%%writefile ../demos/trashify_object_detector/README.md
---
title: Trashify Demo Video! using RT-DETR-V2 🚮
emoji: 🗑️
colorFrom: purple
colorTo: blue
sdk: gradio
sdk_version: 6.21.0
app_file: app.py
pinned: false
license: apache-2.0
---

# 🚮 Trashify Object Detector V4

Object detection demo to detect `trash`, `bin`, `hand`, `trash_arm`, `not_trash`, `not_bin`, `not_hand`.

Used as example for encouraging people to cleanup their local area.

If `trash`, `hand`, `bin` all detected = +1 point.

## Dataset

All Trashify models are trained on a custom hand-labelled dataset of people picking up trash and placing it in a bin.

The dataset can be found on Hugging Face as [`mrdbourke/trashify_manual_labelled_images`](https://huggingface.co/datasets/mrdbourke/trashify_manual_labelled_images).

## Demos

* [V1](https://huggingface.co/spaces/mrdbourke/trashify_demo_v1) = Fine-tuned [Conditional DETR](https://huggingface.co/docs/transformers/en/model_doc/conditional_detr) model trained *without* data augmentation.
* [V2](https://huggingface.co/spaces/mrdbourke/trashify_demo_v2) = Fine-tuned Conditional DETR model trained *with* data augmentation.
* [V3](https://huggingface.co/spaces/mrdbourke/trashify_demo_v3) = Fine-tuned Conditional DETR model trained *with* data augmentation (same as V2) with an NMS (Non Maximum Suppression) post-processing step.
* [V4](https://huggingface.co/spaces/mrdbourke/trashify_demo_v4) = Fine-tuned [RT-DETRv2](https://huggingface.co/docs/transformers/main/en/model_doc/rt_detr_v2) model trained *without* data augmentation or NMS post-processing (current best mAP).

## Learn more

See the full end-to-end code of how this demo was built at [learnhuggingface.com](https://www.learnhuggingface.com/notebooks/hugging_face_object_detection_tutorial).

Writing ../demos/trashify_object_detector/README.md


In [24]:
gr.__version__

'6.21.0'

### Making an examples folder

Having examples in our app is a great way for people to understand the premise straight away.

In [26]:
from pathlib import Path

demo_example_dir = "../demos/trashify_object_detector/trashify_examples/"
demo_example_dir_path = Path(demo_example_dir)
demo_example_dir_path.mkdir(parents=True, exist_ok=True)

In [31]:
from datasets import load_dataset

trashify_examples = load_dataset("mrdbourke/trashify_examples")

In [30]:
trashify_examples

DatasetDict({
    train: Dataset({
        features: ['image'],
        num_rows: 3
    })
})

In [32]:
for i, sample in enumerate(trashify_examples["train"]):
  save_path = Path(demo_example_dir_path / f"trashify_example_{i}.jpeg")
  print(f"[INFO] Saving image to: {save_path}")
  sample["image"].save(save_path)

[INFO] Saving image to: ../demos/trashify_object_detector/trashify_examples/trashify_example_0.jpeg
[INFO] Saving image to: ../demos/trashify_object_detector/trashify_examples/trashify_example_1.jpeg
[INFO] Saving image to: ../demos/trashify_object_detector/trashify_examples/trashify_example_2.jpeg


### Uploading our demo to Hugging Face Spaces

To upload our demo to Hugging Face Spaces, we'll use the Hugging Face Hub PyThon API:

1. Import the required methods from `huggingface_hub` package. https://huggingface.co/docs/huggingface_hub/v0.9.1/en/package_reference/hf_api
2. Define the demo folder we'd like to upload and different parameters our space, e.g. the SDK to use.
3. Create a repo with Hugging Face Spaces with `create_repo` method.
4. Get the full name of our repo so we can reference it.
5. Upload the contents of our demo app to the repo we made.
6. Hope it all works and inspect results!

In [41]:
import huggingface_hub

In [43]:
# 1. Import the required methods for uploading to the Hugging Face Hub
from huggingface_hub import (
    create_repo,
    get_full_repo_name,
    upload_file, # for uploading a single file (if necessary)
    upload_folder # for uploading multiple files (in a folder)
)

# 2. Define the parameters we'd like to use for the upload
LOCAL_DEMO_FOLDER_PATH_TO_UPLOAD = "../demos/trashify_object_detector"
HF_TARGET_SPACE_NAME = "trashify_demo_video"
HF_REPO_TYPE = "space"
HF_SPACE_SDK = "gradio"
SPACE_HARDWARE = "zero-a10g" # For free services, See this link here: https://github.com/huggingface/huggingface_hub/blob/v1.25.1/src/huggingface_hub/_space_api.py#L68
HF_TOKEN = "" # Note: you can set this here or just have it part of your environment variables

# 3. Create a Space repository on Hugging Face Hub
# https://huggingface.co/docs/huggingface_hub/v1.25.1/en/package_reference/hf_api#huggingface_hub.HfApi.create_repo
print(f"[INFO] Creating repo on Hugging Face Hub with name: {HF_TARGET_SPACE_NAME}")
create_repo(
    repo_id=HF_TARGET_SPACE_NAME,
    # token=HF_TOKEN,
    repo_type=HF_REPO_TYPE,
    private=False,
    space_sdk=HF_SPACE_SDK,
    exist_ok=True,
    space_hardware = SPACE_HARDWARE
)

# 4. Get the full repository name (e.g. {username}/{model_id} or {username}/{space_name})
# https://huggingface.co/docs/huggingface_hub/v1.25.1/en/package_reference/hf_api#huggingface_hub.HfApi.get_full_repo_name
full_hf_repo_name = get_full_repo_name(model_id=HF_TARGET_SPACE_NAME)
print(f"[INFO] Full Hugging Face Hub repo name: {full_hf_repo_name}")

# 5. Upload our demo folder
# https://huggingface.co/docs/huggingface_hub/v1.25.1/en/package_reference/hf_api#huggingface_hub.HfApi.upload_folder
print(f"[INFO] Uploading {LOCAL_DEMO_FOLDER_PATH_TO_UPLOAD} to repo: {full_hf_repo_name}")
folder_upload_url = upload_folder(
    repo_id=full_hf_repo_name,
    folder_path=LOCAL_DEMO_FOLDER_PATH_TO_UPLOAD,
    path_in_repo=".", # upload our folder to the root directory ("." means "base" or "root", this is the default)
    # token=HF_TOKEN, # optional: set token manually
    repo_type=HF_REPO_TYPE,
    commit_message="Uploading Trashify box detection model app.py"
)
print(f"[INFO] Demo folder successfully uploaded with commit URL: {folder_upload_url}")

[INFO] Creating repo on Hugging Face Hub with name: trashify_demo_video
[INFO] Full Hugging Face Hub repo name: juanpajedrez/trashify_demo_video
[INFO] Uploading ../demos/trashify_object_detector to repo: juanpajedrez/trashify_demo_video
[INFO] Demo folder successfully uploaded with commit URL: https://huggingface.co/spaces/juanpajedrez/trashify_demo_video/commit/088e746c4762db001b2e7d85e1deef671f49177f


### Testing the hosted demo

In [46]:
from IPython.display import display, HTML

display(HTML(data = '''<iframe
	src="https://juanpajedrez-trashify-demo-video.hf.space"
	frameborder="0"
	width="850"
	height="450"
></iframe>'''))
